In [1]:
import os

# Configuración de carpetas destino
BASE_DIR = "."  # Directorio actual
REAL_MINI_DIR = os.path.join(BASE_DIR, "Real_mini")
FAKE_MINI_DIR = os.path.join(BASE_DIR, "Fake_mini")

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')
CATEGORIES = ["deepfakes", "face2face", "faceshifter", "faceswap", "neuraltextures"]

registros_recreados = []

# -------------------------------------------------------------
# 1. Reconstruir registros de imágenes 'Real' (Etiqueta 0)
# -------------------------------------------------------------
if os.path.exists(REAL_MINI_DIR):
    # Listar y ordenar para mantener consistencia
    real_files = sorted([f for f in os.listdir(REAL_MINI_DIR) if f.lower().endswith(VALID_EXTENSIONS)])
    
    for filename in real_files:
        file_path = os.path.abspath(os.path.join(REAL_MINI_DIR, filename))
        registros_recreados.append((0, 'real', file_path))

# -------------------------------------------------------------
# 2. Reconstruir registros de imágenes 'Fake' (Etiqueta 1)
# -------------------------------------------------------------
if os.path.exists(FAKE_MINI_DIR):
    fake_files = sorted([f for f in os.listdir(FAKE_MINI_DIR) if f.lower().endswith(VALID_EXTENSIONS)])
    
    for filename in fake_files:
        file_path = os.path.abspath(os.path.join(FAKE_MINI_DIR, filename))
        
        # Identificar qué categoría es según la parte inicial del nombre del archivo (ej. "deepfakes_001.jpg")
        cat_encontrada = 'unknown'
        filename_lower = filename.lower()
        
        for cat in CATEGORIES:
            if filename_lower.startswith(cat):
                cat_encontrada = cat
                break
                
        registros_recreados.append((1, cat_encontrada, file_path))

# -------------------------------------------------------------
# 3. Convertir a Tupla Final
# -------------------------------------------------------------
dataset_tuple = tuple(registros_recreados)

print(f"✓ Tupla recreada con éxito. Total de elementos: {len(dataset_tuple)}")

# Mostrar los primeros 5 ejemplos para verificar la estructura
print("\nPrimeros 5 elementos de la tupla:")
for elem in dataset_tuple[:5]:
    print(elem)

✓ Tupla recreada con éxito. Total de elementos: 1000

Primeros 5 elementos de la tupla:
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_001.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_002.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_003.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_004.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_005.jpg')


In [2]:
import json

# Guardar la tupla en un archivo JSON
with open("dataset_tuple.json", "w", encoding="utf-8") as f:
    json.dump(dataset_tuple, f, ensure_ascii=False, indent=4)

In [ ]:
import requests
import json
import os
import time

# Configuración de credenciales de Sightengine
API_USER = ''
API_SECRET = ''

# Archivos de entrada y salida
ORIGINAL_JSON = "respuestas_modelos.json"
OUTPUT_JSON = "respuestas_sightengine.json"

# Configuración de reintentos por Límite de Uso / Throttling
MAX_RETRIES = 5         # Número máximo de intentos si da error de tasa/throttling
INITIAL_WAIT_TIME = 2   # Segundos iniciales a esperar si se detecta uso límite

# 1. Cargar el dataset con las rutas de las imágenes
if os.path.exists("dataset_tuple.json"):
    with open("dataset_tuple.json", "r", encoding="utf-8") as f:
        dataset_tuple = json.load(f)
    print(f"✓ Cargadas {len(dataset_tuple)} imágenes desde 'dataset_tuple.json'")
else:
    print("❌ Error: No se encontró el archivo 'dataset_tuple.json'.")
    dataset_tuple = []

# 2. Cargar 'respuestas_modelos' o reanudar desde 'respuestas_sightengine.json'
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)
    print(f"✓ Reanudando desde archivo existente '{OUTPUT_JSON}' ({len(respuestas_modelos)} respuestas cargadas)")
elif os.path.exists(ORIGINAL_JSON):
    with open(ORIGINAL_JSON, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)
    print(f"✓ Cargas iniciales tomadas desde '{ORIGINAL_JSON}'")
else:
    respuestas_modelos = [[None, None, None] for _ in range(len(dataset_tuple))]

# Ajustar tamaño si hay desajuste entre los JSONs
if len(respuestas_modelos) < len(dataset_tuple):
    for _ in range(len(dataset_tuple) - len(respuestas_modelos)):
        respuestas_modelos.append([None, None, None])

def es_error_throttling(res_dict):
    """Comprueba si la respuesta contiene el error de límite de uso (usage_limit / code 33)."""
    if isinstance(res_dict, dict) and res_dict.get("status") == "failure":
        error_info = res_dict.get("error", {})
        if error_info.get("type") == "usage_limit" or error_info.get("code") == 33:
            return True
    return False

def es_respuesta_invalida_o_error(res_dict):
    """Verifica si la respuesta requiere ser procesada nuevamente."""
    if res_dict is None:
        return True
    if isinstance(res_dict, dict):
        if "error" in res_dict or "error_code" in res_dict or res_dict.get("status") == "failure":
            return True
    return False

# 3. Procesar iterativamente utilizando Sightengine
if len(dataset_tuple) > 0:
    print(f"\nIniciando/reanudando evaluación con Sightengine...\n")

    params = {
        'models': 'deepfake',
        'api_user': API_USER,
        'api_secret': API_SECRET
    }

    for idx, item in enumerate(dataset_tuple):
        clasificacion, tipo_fake, file_path = item
        
        # Posición 2 de la tupla (índice 1 en listas base-0) -> Sightengine
        res_sightengine_actual = respuestas_modelos[idx][1]

        # Omitir únicamente si la respuesta actual es totalmente válida
        if not es_respuesta_invalida_o_error(res_sightengine_actual):
            print(f"[{idx+1}/{len(dataset_tuple)}] Omitiendo (ya procesado): {os.path.basename(file_path)}")
            continue

        print(f"[{idx+1}/{len(dataset_tuple)}] Procesando Sightengine: {os.path.basename(file_path)} ({tipo_fake})")

        if not os.path.exists(file_path):
            respuestas_modelos[idx][1] = {"error": f"El archivo no existe en la ruta: {file_path}"}
            continue

        # Bucle de intentos con gestión de Throttling/Límite de peticiones
        intentos = 0
        tiempo_espera = INITIAL_WAIT_TIME

        while intentos < MAX_RETRIES:
            try:
                with open(file_path, 'rb') as f:
                    files = {'media': f}

                    response = requests.post(
                        'https://api.sightengine.com/1.0/check.json',
                        files=files,
                        data=params
                    )

                    if response.status_code == 200:
                        res_sightengine = response.json()
                    else:
                        try:
                            res_sightengine = response.json()
                        except Exception:
                            res_sightengine = {
                                "error_code": response.status_code, 
                                "message": response.text
                            }

            except Exception as e:
                res_sightengine = {"error": str(e)}

            # Detectar si fue bloqueado por límite de peticiones/segundo
            if es_error_throttling(res_sightengine):
                intentos += 1
                print(f"  ⚠️ Límite de peticiones alcanzado (Usage Limit/33). Reintentando ({intentos}/{MAX_RETRIES}) en {tiempo_espera}s...")
                time.sleep(tiempo_espera)
                tiempo_espera *= 2  # Incremento exponencial del tiempo de espera
            else:
                # Si no es un error de throttling (ya sea éxito u otro fallo), rompemos el bucle
                break

        # Asignar la respuesta obtenida
        respuestas_modelos[idx][1] = res_sightengine

        # Guardado preventivo cada 10 iteraciones
        if (idx + 1) % 10 == 0:
            with open(OUTPUT_JSON, "w", encoding="utf-8") as out_file:
                json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)

        # Pausa ligera entre solicitudes exitosas
        time.sleep(0.2)

    print(f"\n✓ Proceso finalizado. Total de elementos procesados: {len(respuestas_modelos)}")

    # 4. Guardar todas las respuestas actualizadas
    with open(OUTPUT_JSON, "w", encoding="utf-8") as out_file:
        json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)
    print(f"✓ Todas las respuestas guardadas correctamente en '{OUTPUT_JSON}'")

else:
    print("❌ No hay datos que procesar.")

✓ Cargadas 1000 imágenes desde 'dataset_tuple.json'
✓ Reanudando desde archivo existente 'respuestas_sightengine.json' (1000 respuestas cargadas)

Iniciando/reanudando evaluación con Sightengine...

[1/1000] Omitiendo (ya procesado): real_001.jpg
[2/1000] Omitiendo (ya procesado): real_002.jpg
[3/1000] Omitiendo (ya procesado): real_003.jpg
[4/1000] Omitiendo (ya procesado): real_004.jpg
[5/1000] Omitiendo (ya procesado): real_005.jpg
[6/1000] Omitiendo (ya procesado): real_006.jpg
[7/1000] Omitiendo (ya procesado): real_007.jpg
[8/1000] Omitiendo (ya procesado): real_008.jpg
[9/1000] Omitiendo (ya procesado): real_009.jpg
[10/1000] Omitiendo (ya procesado): real_010.jpg
[11/1000] Omitiendo (ya procesado): real_011.jpg
[12/1000] Omitiendo (ya procesado): real_012.jpg
[13/1000] Omitiendo (ya procesado): real_013.jpg
[14/1000] Omitiendo (ya procesado): real_014.jpg
[15/1000] Omitiendo (ya procesado): real_015.jpg
[16/1000] Omitiendo (ya procesado): real_016.jpg
[17/1000] Omitiendo (ya pr

In [6]:
import json
import os

# Priorizamos evaluar el archivo generado para Sightengine
FILENAME_JSON = "respuestas_sightengine.json" if os.path.exists("respuestas_sightengine.json") else "respuestas_modelos.json"
FILENAME_DATASET = "dataset_tuple.json"

def es_respuesta_valida_sightengine(respuesta):
    """Comprueba si la respuesta de Sightengine es válida y contiene valores reales."""
    if respuesta is None:
        return False, "Es None (Sin procesar)"
    
    if not isinstance(respuesta, dict):
        return False, f"Formato no válido ({type(respuesta).__name__})"
    
    # Comprobar errores genéricos, fallos de API o códigos HTTP
    if "error" in respuesta or "error_code" in respuesta:
        error_msg = respuesta.get("error") or respuesta.get("message") or respuesta.get("detail")
        return False, f"Error devuelto: {error_msg}"
        
    if respuesta.get("status") == "failure":
        error_det = respuesta.get("error", {}).get("message", "Sin detalle")
        return False, f"Status: failure ({error_det})"
    
    # Validación específica de la payload de Sightengine
    # La API de Sightengine suele responder con status: 'success' y la clave 'type' -> 'deepfake'
    if respuesta.get("status") == "success":
        type_data = respuesta.get("type", {})
        if "deepfake" in type_data and isinstance(type_data["deepfake"], (int, float)):
            return True, f"OK (Valor deepfake: {type_data['deepfake']})"
        elif "type" not in respuesta:
            return False, "Estructura incompleta: Falta el campo 'type'"
            
    return True, "OK"

def verificar_sightengine():
    if not os.path.exists(FILENAME_JSON):
        print(f"❌ Error: No se encontró el archivo '{FILENAME_JSON}'.")
        return

    # Cargar respuestas
    with open(FILENAME_JSON, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)

    # Cargar dataset si existe para vincular los nombres de archivo
    dataset_tuple = []
    if os.path.exists(FILENAME_DATASET):
        with open(FILENAME_DATASET, "r", encoding="utf-8") as f:
            dataset_tuple = json.load(f)

    print(f"🔍 Evaluando Sightengine (Posición 2 / Índice 1) en '{FILENAME_JSON}'...")
    print(f"📋 Total de registros a revisar: {len(respuestas_modelos)}\n")
    print("=" * 80)
    
    total_registros = len(respuestas_modelos)
    errores_pos2 = 0
    correctos_pos2 = 0

    for idx, triada in enumerate(respuestas_modelos):
        # Asegurar longitud mínima de la lista
        while len(triada) < 2:
            triada.append(None)

        # Evaluar la posición 2 (índice 1 -> Sightengine)
        valido, motivo = es_respuesta_valida_sightengine(triada[1])

        # Nombre del archivo origen si está disponible
        nombre_archivo = ""
        if idx < len(dataset_tuple):
            nombre_archivo = f" | Archivo: {os.path.basename(dataset_tuple[idx][2])}"

        if not valido:
            errores_pos2 += 1
            print(f"❌ Índice {idx}{nombre_archivo}")
            print(f"   └─ Detalle: {motivo}\n")
        else:
            correctos_pos2 += 1

    print("=" * 80)
    print("\n📊 RESUMEN SIGHTENGINE (POSICIÓN 2):")
    print(f"• Archivo analizado: {FILENAME_JSON}")
    print(f"• Total analizado:   {total_registros}")
    print(f"• Válidos (✓):       {correctos_pos2} / {total_registros}")
    print(f"• Errores/Faltantes: {errores_pos2} / {total_registros}")

if __name__ == "__main__":
    verificar_sightengine()

🔍 Evaluando Sightengine (Posición 2 / Índice 1) en 'respuestas_sightengine.json'...
📋 Total de registros a revisar: 1000


📊 RESUMEN SIGHTENGINE (POSICIÓN 2):
• Archivo analizado: respuestas_sightengine.json
• Total analizado:   1000
• Válidos (✓):       1000 / 1000
• Errores/Faltantes: 0 / 1000
